
## Customer Retention & RFM Analysis

### 🛠️ Tools & Technologies Used
Python
Pandas
NumPy
MySQL

## Business Questions 

1.	What percentage of our customers are Champions, and how much revenue do they generate? Are we dangerously dependent on a small group?
2.	Which customers were high-value but have gone quiet recently? How much revenue is at risk if they churn?
3.	What does the cohort retention curve look like? Of customers who first bought in January, what % were still buying in March, June, and December?
4.	Is our Pareto distribution healthy? Do 20% of customers drive 80% of revenue — and is that concentration rising or falling?
5.	What specific action should each customer segment receive this week? Not in theory — in practice.


In [4]:
### Step 1: Import Libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Load Dataset

In [5]:
df = pd.read_csv("online_retail_II.csv")

In [6]:
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


### Dataset Overview 

In [7]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [8]:
df.tail()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.00,12680.0,France


In [9]:
### Observation 
# The dataset contains transaction level retail purchase records from an online retail store.

In [10]:
print(df.columns)

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')


In [11]:
### Observation 

# The dataset contain 8 columns named 'Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country'

#### Dataset shape

In [12]:
print(df.shape)

(1067371, 8)


In [13]:
### Observation 

# The dataset contain 1067371 rows and 8 columns.

#### Data Types & Missing Value

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


In [15]:
print(df.dtypes)

Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object


In [16]:
### Observations

# InvoiceDate is currently stored as object datatype and will require conversion to datetime format for time based analysis such as recency and cohort analysis.

# Customer ID is stored as float datatype and may require conversion to integer after handling missing values for better consistency and readability. 

In [17]:
print(df['InvoiceDate'].min(), df['InvoiceDate'].max())

2009-12-01 07:45:00 2011-12-09 12:50:00


In [18]:
### Observation

# Dataset spans Dec 2009 to Dec 2011 — 2 years of data

In [19]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [20]:
### Observations

# Description and Customer ID columns contain missing values.
# Missing Customer IDs may affect customer level analysis such as RFM segmentation and retention analysis

In [21]:
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(null_pct)

Invoice         0.00
StockCode       0.00
Description     0.41
Quantity        0.00
InvoiceDate     0.00
Price           0.00
Customer ID    22.77
Country         0.00
dtype: float64


In [22]:
# Observation

# CustomerID has 24% nulls  significant for RFM

#### Statistical Summary

In [23]:
df.describe()

,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


In [24]:
print("Cancellations:", df[df['Quantity'] < 0].shape[0])
print("Negative price rows:", df[df['Price'] <= 0].shape[0])

Cancellations: 22950
Negative price rows: 6207


In [25]:
#### Observations

# The dataset contains over 1 million transaction records, making it suitable for large scale customer and retention analysis.
# Negative values are present in both Quantity and Price columns, likely representing returned or cancelled transactions that will require cleaning before revenue analysis.
# The large gap between median and maximum values in Quantity and Price suggests the presence of outliers and highly skewed purchasing behavior.
# Most transactions involve relatively small purchase quantities, as the median quantity is only 3 despite extremely large maximum values.
# Missing Customer IDs may impact customer-level analysis such as RFM segmentation and cohort retention analysis.

In [26]:
df.nunique()

Invoice        53628
StockCode       5305
Description     5698
Quantity        1057
InvoiceDate    47635
Price           2807
Customer ID     5942
Country           43
dtype: int64

In [27]:
### Observations
# The dataset contains a very large number of unique invoices and customer IDs, indicating a broad transactional customer base suitable for retention and RFM analysis.
# A high number of unique stock codes and product descriptions suggests a diverse product catalog with wide product variety.
# The dataset includes transactions from 43 countries, making it possible to perform geographic revenue and customer analysis.
# The large number of unique invoice dates indicates high transaction frequency over time, which is useful for cohort and retention analysis.
# The relatively smaller number of unique quantities and prices compared to invoices suggests repeated purchasing patterns across products.

#### Column Overview

In [28]:
cols = []
for col in df.columns:
    sample_vals = df[col].dropna().astype(str).unique()[:5].tolist()
    cols.append({
        "column": col,
        "dtype": df[col].dtype,
        "missing": df[col].isnull().sum(),
        "unique": df[col].nunique(dropna=True),
        "sample_values": sample_vals
    })

col_summary = pd.DataFrame(cols)
col_summary

,column,dtype,missing,unique,sample_values
0,Invoice,object,0,53628,"[489434, 489435, 489436, 489437, 489438]"
1,StockCode,object,0,5305,"[85048, 79323P, 79323W, 22041, 21232]"
2,Description,object,4382,5698,"[15CM CHRISTMAS GLASS BALL 20 LIGHTS, PINK CHE..."
3,Quantity,int64,0,1057,"[12, 48, 24, 10, 18]"
4,InvoiceDate,object,0,47635,"[2009-12-01 07:45:00, 2009-12-01 07:46:00, 200..."
5,Price,float64,0,2807,"[6.95, 6.75, 2.1, 1.25, 1.65]"
6,Customer ID,float64,243007,5942,"[13085.0, 13078.0, 15362.0, 18102.0, 12682.0]"
7,Country,object,0,43,"[United Kingdom, France, USA, Belgium, Australia]"


In [29]:
### Overall Initial Observations

# The dataset contains large scale transactional retail purchase data suitable for customer retention and RFM analysis.

# Missing Customer IDs and Description values indicate data quality issues that will require cleaning before customer level analysis.

# Negative Quantity and Price values likely represent returned or cancelled transactions and may affect revenue calculations.

# InvoiceDate is currently stored as object datatype and will require datetime conversion for time-based analysis such as cohort and recency analysis.

# The dataset contains a diverse product catalog and customers from multiple countries, making it suitable for segmentation and retention analysis.

#### Cleaning and Handling Missing Values


In [30]:
df.head(3)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom


In [31]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [32]:
df[df["Customer ID"].isnull()]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
1066997,581498,85099B,JUMBO BAG RED RETROSPOT,5,2011-12-09 10:26:00,4.13,NaN,United Kingdom
1066998,581498,85099C,JUMBO BAG BAROQUE BLACK WHITE,4,2011-12-09 10:26:00,4.13,NaN,United Kingdom
1066999,581498,85150,LADIES & GENTLEMEN METAL SIGN,1,2011-12-09 10:26:00,4.96,NaN,United Kingdom
1067000,581498,85174,S/4 CACTI CANDLES,1,2011-12-09 10:26:00,10.79,NaN,United Kingdom


In [33]:
## Observations

# Negative Quantity values and zero Price entries strongly suggest that these records represent cancelled orders, returned products, or adjustment transactions.

# Several rows with missing Customer IDs also contain unusual descriptions, negative quantities, or zero-priced transactions, indicating that many of these records may not represent valid customer purchases.

# Customer ID is the primary grouping key for all RFM calculations. Transactions without Customer IDs cannot be reliably linked to individual customers and may distort customer retention and segmentation analysis.

# Therefore, rows with missing Customer IDs will be removed during the cleaning process to maintain accurate customer-level analysis.

In [34]:
# Store original row count
before = df.shape[0]

print("Rows before cleaning:", before)

# Remove rows with missing Customer IDs
df_clean = df.dropna(subset=["Customer ID"])

# Remove cancelled/returned transactions
df_clean = df_clean[df_clean["Quantity"] > 0]

# Remove invalid price rows
df_clean = df_clean[df_clean["Price"] > 0]

# Store cleaned row count
after = df_clean.shape[0]

print("Rows after cleaning:", after)

# Calculate percentage dropped
dropped_percent = ((before - after) / before) * 100

print(f"Percentage of rows dropped: {dropped_percent:.2f}%")

Rows before cleaning: 1067371
Rows after cleaning: 805549
Percentage of rows dropped: 24.53%


In [35]:
## Observations

# Rows with missing Customer IDs were removed because customer level tracking is essential for RFM and retention analysis.

# Transactions with negative Quantity values were removed as they likely represent cancelled or returned orders.

# Rows with zero or negative Price values were removed because they do not represent valid revenue-generating transactions.

# Approximately 24.53% of rows were removed during the cleaning process to improve the reliability and accuracy of customer behavior analysis.

#### Standardizing Column Names¶



In [36]:
df_clean.columns = df_clean.columns.str.strip().str.lower().str.replace(" ", "_")

In [37]:
### Observation

# Standardized column names by converting them to lowercase
# and replacing spaces with underscores for better consistency and easier querying during analysis.

In [38]:
# converting invoicedate to datetime format for time based analysis such as recency and cohort analysis.

df_clean['invoicedate'] = pd.to_datetime(df_clean['invoicedate'])
df_clean["customer_id"] =  df_clean["customer_id"].astype(int)

In [39]:
df_clean["revenue"] = df_clean["quantity"] * df_clean["price"]

In [40]:
### Observation 

# Created revenue column to support monetary and customer spending analysis.

In [41]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 805549 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   invoice      805549 non-null  object        
 1   stockcode    805549 non-null  object        
 2   description  805549 non-null  object        
 3   quantity     805549 non-null  int64         
 4   invoicedate  805549 non-null  datetime64[ns]
 5   price        805549 non-null  float64       
 6   customer_id  805549 non-null  int64         
 7   country      805549 non-null  object        
 8   revenue      805549 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(2), object(4)
memory usage: 61.5+ MB


In [42]:
# Check duplicate rows

duplicate_count = df_clean.duplicated().sum()

print("Duplicate rows:", duplicate_count)

# Remove exact duplicate rows
df_clean.drop_duplicates(inplace=True)

# Verify duplicates removed
print("Duplicate rows found:", duplicate_count)
print("Remaining duplicates:", df_clean.duplicated().sum())


Duplicate rows: 26124
Duplicate rows found: 26124
Remaining duplicates: 0


In [43]:
## Observations

# Exact duplicate transactional rows were identified and removed to avoid double counting revenue, purchase frequency,
# and customer activity during RFM analysis.
# Removing duplicate records improves the reliability and accuracy of customer-level insights.

In [44]:
print(df_clean['invoicedate'].min())
print(df_clean['invoicedate'].max())
# Important — cancellations could skew the date range

2009-12-01 07:45:00
2011-12-09 12:50:00


In [45]:
print(df_clean.shape)
print(df_clean.dtypes)
print(df_clean.isnull().sum())
df_clean.head()

(779425, 9)
invoice                object
stockcode              object
description            object
quantity                int64
invoicedate    datetime64[ns]
price                 float64
customer_id             int64
country                object
revenue               float64
dtype: object
invoice        0
stockcode      0
description    0
quantity       0
invoicedate    0
price          0
customer_id    0
country        0
revenue        0
dtype: int64


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [95]:
## Cleaning Summary

# Removed rows with missing customer IDs
# Removed cancelled and invalid transactions
# Removed duplicate records
# Standardized column names
# Converted invoice_date to datetime datatype
# Converted customer_id to integer datatype
# Created revenue feature for monetary analysis

# Dataset is now cleaned and ready for RFM analysis.

In [96]:
# Save cleaned data from Python
df_clean.to_csv("online_retail_clean.csv", index=False)
print(f"Saved {len(df_clean):,} rows to CSV")

Saved 779,425 rows to CSV


In [98]:
df_clean

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00
...,...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,12.60
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,16.60
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,16.60
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680,France,14.85


In [99]:
pip install sqlalchemy

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [100]:
%pip install sqlalchemy pymysql

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [101]:
database = "customer_retention"

In [102]:
from sqlalchemy import create_engine
import pandas as pd

# Define credentials
username = "root"
password = "aafreen78"
host = "localhost"
port = "3306"
database = "customer_retention"

# Create engine
engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

# Table name
table_name = "orders"

# Upload CLEAN dataframe
df_clean.to_sql(table_name, engine, if_exists="replace", index=False)

# Check data
pd.read_sql("SELECT * FROM orders LIMIT 5;", engine)

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [ ]:


rfm = pd.read_csv('rfm_segments.csv')

table_name = "rfm_segments"

rfm.to_sql(
    name=table_name,
    con=engine,
    if_exists="replace",
    index=False
)

print("RFM Segment table uploaded successfully!")